# Lecture 20 — How Do We Know If Our Model Really Learned?

The blog is the textbook; this notebook is the laboratory. Predict before running each experiment.

▶️ Run in Colab: https://colab.research.google.com/github/manish7725/deeplearning/blob/main/Lecture%2020%20-%20How%20Do%20We%20Know%20If%20Our%20Model%20Really%20Learned/notebook.ipynb


## 1. Problem

A model can fit training examples without generalizing. We will compare training and held-out behavior as tree complexity changes.


## 2. Prediction

As a decision tree gets deeper, predict what should happen to training accuracy and validation accuracy.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

rng = np.random.default_rng(7)
X = rng.normal(size=(300, 2))
y = ((X[:,0]**2 + X[:,1]**2 + 0.35*X[:,0]) > 1.1).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=7, stratify=y)
depths = [1, 2, 3, 5, 10, None]
train_scores, test_scores = [], []
for depth in depths:
    model = DecisionTreeClassifier(max_depth=depth, random_state=7)
    model.fit(X_train, y_train)
    train_scores.append(model.score(X_train, y_train))
    test_scores.append(model.score(X_test, y_test))
print('depths:', depths)
print('train:', train_scores)
print('test :', test_scores)


## 4. Mathematics

Accuracy = $(TP+TN)/(TP+TN+FP+FN)$. Precision = $TP/(TP+FP)$. Recall = $TP/(TP+FN)$. F1 = $2PR/(P+R)$.


In [ ]:
tp, fp, fn, tn = 8, 2, 4, 6
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)
assert np.isclose(precision, 0.8)
assert np.isclose(recall, 2/3)
assert np.isclose(f1, 0.7272727272727273)
print('precision:', precision)
print('recall:', recall)
print('f1:', f1)


In [ ]:
# First implementation: metrics from raw predictions.
y_true = np.array([1,1,1,1,0,0,0,0,0,0,0,0])
y_pred = np.array([1,1,1,0,1,0,0,0,0,0,0,0])
cm = confusion_matrix(y_true, y_pred)
tn2, fp2, fn2, tp2 = cm.ravel()
p2 = tp2/(tp2+fp2)
r2 = tp2/(tp2+fn2)
f12 = 2*p2*r2/(p2+r2)
assert np.isclose(p2, precision_score(y_true, y_pred))
assert np.isclose(r2, recall_score(y_true, y_pred))
assert np.isclose(f12, f1_score(y_true, y_pred))
print('confusion matrix:\n', cm)
print('manual:', p2, r2, f12)


In [ ]:
# Visualization: learning curve across tree depth.
plt.figure()
x = np.arange(len(depths))
plt.plot(x, train_scores, marker='o', label='train')
plt.plot(x, test_scores, marker='o', label='held-out')
plt.xticks(x, [str(d) for d in depths])
plt.xlabel('max_depth')
plt.ylabel('accuracy')
plt.title('Model complexity vs generalization')
plt.legend()
plt.show()


In [ ]:
# Controlled experiment: change exactly one variable.
max_depth = 3  # YOUR CHANGE HERE
model = DecisionTreeClassifier(max_depth=max_depth, random_state=7)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('train accuracy:', accuracy_score(y_train, model.predict(X_train)))
print('held-out accuracy:', accuracy_score(y_test, pred))


## 10. Observe

Compare the train and held-out curves. Look for the region where extra complexity stops helping unseen data.


## 11. Explain

Training performance measures fit to seen examples. Held-out performance provides evidence about generalization.


In [ ]:
# Challenge
# Level 4: change only max_depth and report precision, recall and F1.
# Level 5: create an imbalanced dataset and explain why accuracy can become misleading.

depth = 5  # YOUR CODE HERE
challenge = DecisionTreeClassifier(max_depth=depth, random_state=7).fit(X_train, y_train)
pred = challenge.predict(X_test)
print('accuracy:', accuracy_score(y_test, pred))
print('precision:', precision_score(y_test, pred, zero_division=0))
print('recall:', recall_score(y_test, pred, zero_division=0))
print('f1:', f1_score(y_test, pred, zero_division=0))


## 13. Reflection

Checklist: explain train/validation/test roles; identify underfitting and overfitting; compute precision/recall/F1; explain class imbalance; list two leakage paths.
